In [7]:
import pandas as pd
df = pd.read_csv("../data/amazon_reviews.csv")
df.head()
df.shape

C:\Users\jagad\AppData\Local\Temp\ipykernel_13396\922620105.py:2: DtypeWarning: Columns (0: name, 1: reviews.didPurchase) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/amazon_reviews.csv")


(34660, 21)

In [8]:
df.columns
# df.isnull().sum()

Index(['id', 'name', 'asins', 'brand', 'categories', 'keys', 'manufacturer',
       'reviews.date', 'reviews.dateAdded', 'reviews.dateSeen',
       'reviews.didPurchase', 'reviews.doRecommend', 'reviews.id',
       'reviews.numHelpful', 'reviews.rating', 'reviews.sourceURLs',
       'reviews.text', 'reviews.title', 'reviews.userCity',
       'reviews.userProvince', 'reviews.username'],
      dtype='str')

In [9]:
df_Clean = df[[
    "name",
    "brand",
    "categories",
    "reviews.date",
    "reviews.rating",
    "reviews.title",
    "reviews.text"
]].copy()

df_Clean["brand"]=df_Clean["brand"].fillna("Unknown brand").str.strip().str.title()
df_Clean["categories"]=df_Clean["categories"].fillna("Unknown category").str.strip().str.lower()
df_Clean["name"]=df_Clean["name"].fillna("Unknown name").str.strip()
df_Clean["reviews.title"] = df_Clean["reviews.title"].fillna("Unknown Title").str.strip()
df_Clean = df_Clean.dropna(subset=["reviews.rating"])
df_Clean=df_Clean.dropna(subset=["reviews.text"])
df_Clean["reviews.date"] = pd.to_datetime(df_Clean["reviews.date"],errors="coerce")

In [10]:
df_Clean["search_text"] = (
    df_Clean["name"].astype(str)
    + " "
    + df_Clean["reviews.title"].astype(str)
    + " "
    + df_Clean["reviews.text"].astype(str)
)

In [11]:
print(df_Clean.shape)
df_Clean.isnull().sum()
df_Clean.to_csv("../data/clean_amazon_reviews.csv", index=False)

(34626, 8)


In [13]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)
sample_text = df_Clean["search_text"].iloc[0]
print(sample_text[:200])
embedding = model.encode(sample_text)
print(type(embedding))
print(len(embedding))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi, 16 GB - Includes Special Offers, Magenta Kindle This product so far has not disappointed. My children love to use it and I like the ability to monitor co
<class 'numpy.ndarray'>
384


In [16]:
subset = df_Clean.head(1000)
embeddings = model.encode(
    subset["search_text"].tolist(),
    show_progress_bar=True
)
print(type(embeddings))
print(embeddings.shape)

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

<class 'numpy.ndarray'>
(1000, 384)


In [ ]:
import chromadb
client = chromadb.Client()
print("Client Created")
collection = client.get_or_create_collection(
    name="amazon_reviews"
)
print("Collection Created")
print(subset.shape)
print(embeddings.shape)
collection.add(
    ids=[str(i) for i in range(len(subset))],
    documents=subset["search_text"].tolist(),
    embeddings=embeddings.tolist(),
    metadatas=[
        {
            "name": str(row["name"]),
            "brand": str(row["brand"]),
            "rating": float(row["reviews.rating"]),
            "category": str(row["categories"])
        }
        for _, row in subset.iterrows()
    ]
)
query = "tablet for kids"
query_embedding = model.encode(query)
results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=5
)


for i in range(len(results["documents"][0])):
    print("=" * 80)
    print("Product:", results["metadatas"][0][i]["name"])
    print("Brand:", results["metadatas"][0][i]["brand"])
    print("Rating:", results["metadatas"][0][i]["rating"])
    print("Review:", results["documents"][0][i][:400])

Client Created
Collection Created
(1000, 8)
(1000, 384)


TypeError: 'NoneType' object is not subscriptable

In [27]:
client.delete_collection("amazon_reviews")

collection = client.get_or_create_collection(
    name="amazon_reviews"
)

print("Fresh Collection Created")

Fresh Collection Created


In [28]:
collection.add(
    ids=[str(i) for i in range(len(subset))],
    documents=subset["search_text"].tolist(),
    embeddings=embeddings.tolist(),
    metadatas=[
        {
            "name": str(row["name"]),
            "brand": str(row["brand"]),
            "rating": float(row["reviews.rating"]),
            "category": str(row["categories"])
        }
        for _, row in subset.iterrows()
    ]
)

print("Inserted with metadata successfully")

Inserted with metadata successfully


In [32]:
query = "tablet for kids"

query_embedding = model.encode(query)

results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=5
)
# for i in range(len(results["documents"][0])):
#     print("=" * 80)
#     print("Product:", results["metadatas"][0][i]["name"])
#     print("Brand:", results["metadatas"][0][i]["brand"])
#     print("Rating:", results["metadatas"][0][i]["rating"])
#     print("Review:", results["documents"][0][i][:400])


for i in range(len(results["documents"][0])):
    
    rating = results["metadatas"][0][i]["rating"]
    
    if rating >= 4:
        print("=" * 80)
        print("Product:", results["metadatas"][0][i]["name"])
        print("Rating:", rating)
        print(results["documents"][0][i][:300])

Product: All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi, 16 GB - Includes Special Offers, Magenta
Rating: 5.0
All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi, 16 GB - Includes Special Offers, Magenta Great tablet for children I previously had bought an older model of these tablets for my children. I wasn't to happy with them. But this tablet is great, it has child safety features like wether you would like to
Product: All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi, 16 GB - Includes Special Offers, Magenta
Rating: 5.0
All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi, 16 GB - Includes Special Offers, Magenta A great starter tablet for a youngster Santa dropped this off for our 4th grader along with expanded memory. he uses it for Netflix and Youtube as well as Minecraft and other Apps. Why spend hundreds on something
Product: All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi, 16 GB - Includes Special Offers, Magenta
Rating: 4.0
All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi, 16 GB - Includes Special